# Notebook 07c — SCTS-v2 Trust Score on Canonical Shared Samples

**Purpose:** Compute per-sample trust scores combining calibration confidence, explanation stability, and conformal coverage. Extends the existing UNSW-only `07_scts_v2.ipynb` to all 3 datasets × 6 model variants using canonical samples from 04c/06v3/05c.

**What is SCTS-v2 (verbatim from existing 07_scts_v2):**

SHAP-Calibrated Trust Score, version 2. A per-alert score in [0, 100] that combines three orthogonal signals:

| Component | What it measures | Source |
|---|---|---|
| **c₁ — Calibration confidence** | Calibrated probability of predicted class | Pre-saved `_test_proba_calibrated.npy` (verified identical to refit hybrid calibrators) |
| **c₂ — Explanation stability** | Worst-case per-sample Jaccard top-10 under adversarial perturbation | `stability_v2_per_sample_jaccard.csv` from 05c |
| **c₃ — Conformal coverage** | Per-sample conformal score at α=0.05 using Mondrian per-class thresholds | Computed inline using Mondrian split-conformal (Vovk 2005, Boström 2020) |

**Combination formula:** SCTS-v2 = (c₁ · c₂ · c₃)^(1/3) · 100

Geometric mean penalises asymmetry: weak in any one component → low SCTS even if others are strong. eps=1e-6 used to avoid log(0).

**Methodology improvements over existing 07_scts_v2:**
1. **Canonical samples:** Uses the 1000 per-dataset shared indices from 04c. Same samples that 06v3 (Krishna) and 05c (stability) used. Eliminates the sample-alignment problem the existing 07 had.
2. **Cleaner conformal split:** All test samples NOT in canonical 1000 serve as the conformal calibration set; canonical 1000 are scored. No sample leakage. More data for the quantile estimate.
3. **All 3 datasets, all 6 model variants:** 18 models total instead of just NSL-or-UNSW.

**Time estimate:** ~5-10 minutes (pure CPU, all data preloaded).

**Outputs:**
- `results/tables/scts_v2_canonical.csv` (per-sample SCTS scores: 18,000 rows)
- `results/tables/scts_v2_per_class_summary.csv` (per-class mean SCTS, c₁, c₂, c₃)
- `results/tables/scts_v2_alpha_sensitivity.csv` (α ∈ {0.05, 0.10, 0.20})
- `results/tables/scts_v2_validation.csv` (SCTS quartile vs accuracy)
- `results/tables/scts_v2_summary.json` (aggregate stats, conformal thresholds)

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
ALPHAS = [0.05, 0.10, 0.20]
ALPHA_PRIMARY = 0.05  # Used for the main SCTS computation
EPS = 1e-6  # Floor for geometric mean

print(f'Scope: {len(DATASETS)} datasets × {len(MODELS_PER_DATASET)} models = {len(DATASETS) * len(MODELS_PER_DATASET)} model-dataset cells')
print(f'Primary alpha for SCTS: {ALPHA_PRIMARY}')
print(f'Sensitivity alphas: {ALPHAS}')

Scope: 1 datasets × 6 models = 6 model-dataset cells
Primary alpha for SCTS: 0.05
Sensitivity alphas: [0.05, 0.1, 0.2]


## 2. Conformal threshold and per-sample c₃

In [3]:
def split_conformal_threshold(calib_probs, y_calib, alpha):
    """Marginal split-conformal threshold (Romano et al. 2019).
    Nonconformity score s_i = 1 - p_hat(y_i | x_i) using TRUE label.
    Threshold = ceil((n+1)(1-alpha)) / n quantile of scores.
    """
    n = len(y_calib)
    scores = 1.0 - calib_probs[np.arange(n), y_calib]
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)
    return float(np.quantile(scores, q_level))

def mondrian_conformal_thresholds(calib_probs, y_calib, alpha, n_classes=5, min_calib=30):
    """Mondrian per-class conformal thresholds (Vovk 2005, Boström 2020).
    Stratifies by PREDICTED class (deployment-feasible since true class unknown at test time).

    For each predicted class c:
      - Find calibration samples where model predicted class c
      - Compute their nonconformity scores: 1 - p_hat(y_true)
      - Take 95th percentile as threshold for that class
      - Fall back to marginal threshold if n_calib_c < min_calib (defaults to 30)

    Returns: dict {class_idx: threshold}, fallback_classes: list of classes that used marginal
    """
    y_pred = calib_probs.argmax(axis=1)
    marginal_thresh = split_conformal_threshold(calib_probs, y_calib, alpha)

    thresholds = {}
    fallback_classes = []
    n_per_class = {}

    for c in range(n_classes):
        mask_c = (y_pred == c)
        n_c = int(mask_c.sum())
        n_per_class[c] = n_c

        if n_c < min_calib:
            thresholds[c] = marginal_thresh
            fallback_classes.append(c)
        else:
            scores_c = 1.0 - calib_probs[mask_c, :][np.arange(n_c), y_calib[mask_c]]
            q_level = min(np.ceil((n_c + 1) * (1 - alpha)) / n_c, 1.0)
            thresholds[c] = float(np.quantile(scores_c, q_level))

    return thresholds, fallback_classes, n_per_class

def empirical_coverage(test_probs, y_test, thresh):
    """Fraction of test samples where 1 - p_hat(y_true) <= threshold (marginal coverage)."""
    n = len(y_test)
    scores = 1.0 - test_probs[np.arange(n), y_test]
    return float((scores <= thresh).mean())

def empirical_coverage_mondrian(test_probs, y_test, thresholds):
    """Per-sample coverage using Mondrian thresholds (lookup by PREDICTED class)."""
    y_pred = test_probs.argmax(axis=1)
    n = len(y_test)
    scores = 1.0 - test_probs[np.arange(n), y_test]
    sample_thresh = np.array([thresholds[int(p)] for p in y_pred])
    return float((scores <= sample_thresh).mean())

def component_3_mondrian(test_probs, y_pred, thresholds):
    """Per-sample c3 with Mondrian per-class thresholds.
    Threshold for each test sample is determined by its predicted class.
    """
    sample_thresh = np.array([thresholds[int(p)] for p in y_pred], dtype=np.float32)
    s = 1.0 - test_probs[np.arange(len(y_pred)), y_pred]
    c3 = np.clip(1.0 - s / sample_thresh, 0.0, 1.0)
    return c3.astype(np.float32)

def component_3(test_probs, y_pred, thresh):
    """(Legacy) Marginal c3 — single threshold."""
    s = 1.0 - test_probs[np.arange(len(y_pred)), y_pred]
    c3 = np.clip(1.0 - s / thresh, 0.0, 1.0)
    return c3.astype(np.float32)

print('✓ Conformal helpers loaded (marginal + Mondrian per-class)')

✓ Conformal helpers loaded (marginal + Mondrian per-class)


## 3. Load per-sample Jaccard (c₂ source) and compute worst-case per sample

In [4]:
# Load stability per-sample Jaccard (from 05c)
stab_path = Path(REPO) / 'results' / 'tables' / 'stability_v2_per_sample_jaccard.csv'
df_stab = pd.read_csv(stab_path)
print(f'Loaded {stab_path.name}: {len(df_stab)} rows')
print(f'Columns: {df_stab.columns.tolist()}')

# Pivot: for each (dataset, model, sample_position) take MIN Jaccard across 3 perturbations
worst_case = df_stab.groupby(['dataset', 'model', 'sample_position'])['jaccard_top10'].min().reset_index()
worst_case.rename(columns={'jaccard_top10': 'worst_jaccard'}, inplace=True)
print(f'\nWorst-case per sample: {len(worst_case)} rows (expect 18 models × 1000 samples = 18000)')
print(f'Mean worst-case Jaccard: {worst_case["worst_jaccard"].mean():.3f}')
print(f'Min: {worst_case["worst_jaccard"].min():.3f}, Max: {worst_case["worst_jaccard"].max():.3f}')

# Build a lookup: (ds, model) -> array of shape (1000,) of worst-case Jaccards
c2_lookup = {}
for (ds, m), g in worst_case.groupby(['dataset', 'model']):
    g_sorted = g.sort_values('sample_position')
    c2_lookup[(ds, m)] = g_sorted['worst_jaccard'].values.astype(np.float32)

print(f'\nc2_lookup populated for {len(c2_lookup)} (dataset, model) combinations')

Loaded stability_v2_per_sample_jaccard.csv: 18000 rows
Columns: ['dataset', 'model', 'perturbation', 'sample_position', 'true_class', 'jaccard_top10']

Worst-case per sample: 6000 rows (expect 18 models × 1000 samples = 18000)
Mean worst-case Jaccard: 0.562
Min: 0.111, Max: 1.000

c2_lookup populated for 6 (dataset, model) combinations


## 4. Main SCTS computation loop

In [5]:
scts_records = []
alpha_records = []
conformal_meta = {}
validation_records = []
per_class_records = []

t_overall = time.time()
print(f'\n{"="*70}')
print(f'SCTS-v2 computation — {datetime.now().strftime("%H:%M:%S")}')
print(f'{"="*70}\n')

for ds in DATASETS:
    print(f'\n=== {ds} ===')

    # Load canonical indices
    canonical_eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    y_test_full = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')

    # Build conformal calibration mask: all test samples EXCEPT canonical 1000
    n_test = len(y_test_full)
    conformal_calib_mask = np.ones(n_test, dtype=bool)
    conformal_calib_mask[canonical_eval_idx] = False
    n_conformal_calib = conformal_calib_mask.sum()

    y_canonical = y_test_full[canonical_eval_idx]
    y_conformal_calib = y_test_full[conformal_calib_mask]

    print(f'  n_test={n_test}, n_conformal_calib={n_conformal_calib}, n_canonical={len(canonical_eval_idx)}')

    for model_name in MODELS_PER_DATASET:
        # Load pre-saved calibrated probabilities (verified identical to refit hybrid calibrator output)
        prob_path = Path(REPO) / 'calibrators' / ds / f'{model_name}_test_proba_calibrated.npy'
        if not prob_path.exists():
            print(f'  ❌ {model_name}: calibrated probs missing')
            continue
        calibrated_proba_full = np.load(prob_path)

        # Split into conformal calib and canonical (test for SCTS)
        calibrated_proba_conformal = calibrated_proba_full[conformal_calib_mask]
        calibrated_proba_canonical = calibrated_proba_full[canonical_eval_idx]

        # ===== c1: calibration confidence on canonical samples =====
        y_pred_canonical = calibrated_proba_canonical.argmax(axis=1)
        c1 = calibrated_proba_canonical[np.arange(len(y_pred_canonical)), y_pred_canonical].astype(np.float32)

        # ===== c2: worst-case Jaccard (from stability_v2_per_sample_jaccard.csv) =====
        if (ds, model_name) not in c2_lookup:
            print(f'  ⚠ {model_name}: no c2 data, skipping')
            continue
        c2 = c2_lookup[(ds, model_name)]  # already (1000,) float32

        # ===== c3: Mondrian per-class conformal at primary alpha =====
        mondrian_thresh, fallback_classes, n_per_class = mondrian_conformal_thresholds(
            calibrated_proba_conformal, y_conformal_calib, ALPHA_PRIMARY, n_classes=5, min_calib=30
        )
        marginal_thresh = split_conformal_threshold(
            calibrated_proba_conformal, y_conformal_calib, ALPHA_PRIMARY
        )
        c3 = component_3_mondrian(calibrated_proba_canonical, y_pred_canonical, mondrian_thresh)

        # Empirical coverage on canonical 1000 (Mondrian uses per-sample threshold lookup)
        emp_coverage = empirical_coverage_mondrian(calibrated_proba_canonical, y_canonical, mondrian_thresh)

        # ===== SCTS-v2 =====
        geo_mean = (np.clip(c1, EPS, 1.0) * np.clip(c2, EPS, 1.0) * np.clip(c3, EPS, 1.0)) ** (1/3)
        scts = (geo_mean * 100).astype(np.float32)

        # ===== Accuracy on canonical (for validation) =====
        correct = (y_pred_canonical == y_canonical).astype(float)

        # ===== Per-sample records =====
        for i in range(len(scts)):
            scts_records.append({
                'dataset': ds, 'model': model_name,
                'sample_position': i,
                'true_class': int(y_canonical[i]),
                'pred_class': int(y_pred_canonical[i]),
                'correct': int(correct[i]),
                'c1': float(c1[i]),
                'c2': float(c2[i]),
                'c3': float(c3[i]),
                'scts': float(scts[i]),
            })

        # ===== Conformal metadata =====
        conformal_meta[f'{ds}/{model_name}'] = {
            'marginal_threshold_alpha_0.05': float(marginal_thresh),
            'mondrian_thresholds_alpha_0.05': {str(c): float(t) for c, t in mondrian_thresh.items()},
            'fallback_classes': [int(c) for c in fallback_classes],
            'n_calib_per_predicted_class': {str(c): int(n) for c, n in n_per_class.items()},
            'empirical_coverage_on_canonical_mondrian': float(emp_coverage),
            'n_conformal_calibration_samples': int(n_conformal_calib),
        }

        # ===== Alpha sensitivity (Mondrian) =====
        for alpha in ALPHAS:
            mondrian_thresh_a, fallback_a, _ = mondrian_conformal_thresholds(
                calibrated_proba_conformal, y_conformal_calib, alpha, n_classes=5, min_calib=30
            )
            marginal_thresh_a = split_conformal_threshold(
                calibrated_proba_conformal, y_conformal_calib, alpha
            )
            c3_a = component_3_mondrian(calibrated_proba_canonical, y_pred_canonical, mondrian_thresh_a)
            geo_a = (np.clip(c1, EPS, 1.0) * np.clip(c2, EPS, 1.0) * np.clip(c3_a, EPS, 1.0)) ** (1/3)
            scts_a = geo_a * 100
            emp_cov_a = empirical_coverage_mondrian(calibrated_proba_canonical, y_canonical, mondrian_thresh_a)
            alpha_records.append({
                'dataset': ds, 'model': model_name, 'alpha': alpha,
                'mean_threshold_across_classes': float(np.mean(list(mondrian_thresh_a.values()))),
                'marginal_threshold_reference': float(marginal_thresh_a),
                'empirical_coverage_mondrian': float(emp_cov_a),
                'n_fallback_classes': len(fallback_a),
                'mean_scts': float(scts_a.mean()),
                'median_scts': float(np.median(scts_a)),
            })

        # ===== SCTS quartile validation =====
        overall_acc = correct.mean()
        pearson_corr = np.corrcoef(scts, correct)[0, 1] if scts.std() > 1e-9 else 0.0
        for lo, hi in [(0, 25), (25, 50), (50, 75), (75, 101)]:
            mask = (scts >= lo) & (scts < hi)
            n = int(mask.sum())
            acc = float(correct[mask].mean()) if n > 0 else float('nan')
            validation_records.append({
                'dataset': ds, 'model': model_name,
                'scts_bin_low': lo, 'scts_bin_high': hi,
                'n': n, 'accuracy': acc,
                'overall_accuracy': float(overall_acc),
                'pearson_corr_scts_correct': float(pearson_corr),
            })

        # ===== Per-class summary =====
        for c, cname in enumerate(CLASS_NAMES_5):
            mask = y_canonical == c
            if mask.sum() > 0:
                per_class_records.append({
                    'dataset': ds, 'model': model_name,
                    'true_class': cname, 'class_idx': c,
                    'n': int(mask.sum()),
                    'mean_scts': float(scts[mask].mean()),
                    'mean_c1': float(c1[mask].mean()),
                    'mean_c2': float(c2[mask].mean()),
                    'mean_c3': float(c3[mask].mean()),
                    'accuracy': float(correct[mask].mean()),
                })

        print(f'  ▶ {model_name:<22} marg_thr={marginal_thresh:.3f}, '
              f'mondrian=[{",".join(f"{mondrian_thresh[c]:.2f}" for c in range(5))}], '
              f'fb={len(fallback_classes)}/5, emp_cov={emp_coverage:.3f}, '
              f'mean SCTS={scts.mean():.1f}, median={np.median(scts):.1f}, '
              f'acc={overall_acc:.3f}, corr={pearson_corr:+.3f}')

elapsed = (time.time() - t_overall) / 60
print(f'\nTotal time: {elapsed:.1f} min')
print(f'Per-sample records: {len(scts_records)}')
print(f'Per-class records: {len(per_class_records)}')
print(f'Alpha sensitivity records: {len(alpha_records)}')


SCTS-v2 computation — 13:31:24


=== nsl_kdd_v2 ===
  n_test=22544, n_conformal_calib=21544, n_canonical=1000
  ▶ rf_5class_cw           marg_thr=1.000, mondrian=[1.00,0.45,1.00,0.49,1.00], fb=1/5, emp_cov=0.971, mean SCTS=82.5, median=87.4, acc=0.624, corr=+0.072
  ▶ xgb_5class_cw          marg_thr=1.000, mondrian=[1.00,0.29,1.00,0.40,1.00], fb=1/5, emp_cov=0.973, mean SCTS=82.0, median=86.3, acc=0.638, corr=+0.120
  ▶ dnn_5class_cw          marg_thr=1.000, mondrian=[1.00,0.44,0.99,0.38,1.00], fb=1/5, emp_cov=0.962, mean SCTS=75.8, median=75.4, acc=0.627, corr=+0.003
  ▶ rf_5class_smote        marg_thr=1.000, mondrian=[1.00,0.47,1.00,0.75,1.00], fb=1/5, emp_cov=0.973, mean SCTS=75.9, median=80.7, acc=0.604, corr=-0.029
  ▶ xgb_5class_smote       marg_thr=1.000, mondrian=[1.00,0.36,1.00,0.44,1.00], fb=1/5, emp_cov=0.979, mean SCTS=78.7, median=81.4, acc=0.641, corr=+0.146
  ▶ dnn_5class_smote       marg_thr=1.000, mondrian=[1.00,0.46,1.00,1.00,1.00], fb=0/5, emp_cov=0.973, mean SCTS=7

## 5. Build DataFrames and save CSVs

In [6]:
df_scts = pd.DataFrame(scts_records)
df_perclass = pd.DataFrame(per_class_records)
df_alpha = pd.DataFrame(alpha_records)
df_validation = pd.DataFrame(validation_records)

out_dir = Path(REPO) / 'results' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)

df_scts.to_csv(out_dir / 'scts_v2_canonical.csv', index=False)
df_perclass.to_csv(out_dir / 'scts_v2_per_class_summary.csv', index=False)
df_alpha.to_csv(out_dir / 'scts_v2_alpha_sensitivity.csv', index=False)
df_validation.to_csv(out_dir / 'scts_v2_validation.csv', index=False)

print(f'✓ Saved 4 CSVs:')
print(f'  scts_v2_canonical.csv ({len(df_scts)} rows)')
print(f'  scts_v2_per_class_summary.csv ({len(df_perclass)} rows)')
print(f'  scts_v2_alpha_sensitivity.csv ({len(df_alpha)} rows)')
print(f'  scts_v2_validation.csv ({len(df_validation)} rows)')

✓ Saved 4 CSVs:
  scts_v2_canonical.csv (6000 rows)
  scts_v2_per_class_summary.csv (30 rows)
  scts_v2_alpha_sensitivity.csv (18 rows)
  scts_v2_validation.csv (24 rows)


## 6. Print headline findings

In [7]:
print('=' * 70)
print('SCTS-v2 HEADLINE FINDINGS')
print('=' * 70)

# Aggregate by (dataset, model)
df_scts['architecture'] = df_scts['model'].apply(
    lambda m: 'rf' if 'rf' in m else ('xgb' if 'xgb' in m else 'dnn')
)
df_scts['variant'] = df_scts['model'].apply(
    lambda m: 'cw' if 'cw' in m else 'smote'
)

print('\n--- Mean SCTS per (dataset, architecture) averaged across variants ---')
for ds in DATASETS:
    sub = df_scts[df_scts['dataset'] == ds]
    pivot = sub.groupby('architecture')['scts'].mean().round(1)
    print(f'  {ds:<18} {dict(pivot)}')

print('\n--- Validation: SCTS quartile vs accuracy ---')
print('(If SCTS is meaningful, higher SCTS bin should have higher accuracy)')
agg_val = df_validation.groupby(['scts_bin_low', 'scts_bin_high']).agg({
    'n': 'sum',
    'accuracy': 'mean',
}).reset_index()
for _, row in agg_val.iterrows():
    print(f'  SCTS [{int(row["scts_bin_low"]):>3},{int(row["scts_bin_high"]):>3}): '
          f'n={int(row["n"]):>5}, mean_acc={row["accuracy"]:.3f}')

print('\n--- Pearson correlation SCTS vs correctness ---')
corr_summary = df_validation.groupby(['dataset', 'model'])['pearson_corr_scts_correct'].first()
print(f'  Mean correlation across 18 models: {corr_summary.mean():+.3f}')
print(f'  Min: {corr_summary.min():+.3f}, Max: {corr_summary.max():+.3f}')
print(f'  Models with corr > 0: {(corr_summary > 0).sum()}/{len(corr_summary)}')

print('\n--- Alpha sensitivity (mean SCTS across all 18 models) ---')
alpha_pivot = df_alpha.groupby('alpha')[['mean_threshold_across_classes', 'empirical_coverage_mondrian', 'mean_scts']].mean().round(3)
print(alpha_pivot)

print('\n--- Per-class SCTS (mean across 18 models, by true class) ---')
perclass_pivot = df_perclass.groupby('true_class').agg({
    'mean_scts': 'mean', 'mean_c1': 'mean', 'mean_c2': 'mean', 'mean_c3': 'mean',
    'accuracy': 'mean', 'n': 'sum',
}).round(3)
print(perclass_pivot.reindex(CLASS_NAMES_5))

SCTS-v2 HEADLINE FINDINGS

--- Mean SCTS per (dataset, architecture) averaged across variants ---
  nsl_kdd_v2         {'dnn': np.float64(76.5), 'rf': np.float64(79.2), 'xgb': np.float64(80.4)}

--- Validation: SCTS quartile vs accuracy ---
(If SCTS is meaningful, higher SCTS bin should have higher accuracy)
  SCTS [  0, 25): n=   72, mean_acc=0.702
  SCTS [ 25, 50): n=   69, mean_acc=0.604
  SCTS [ 50, 75): n=  999, mean_acc=0.422
  SCTS [ 75,101): n= 4860, mean_acc=0.662

--- Pearson correlation SCTS vs correctness ---
  Mean correlation across 18 models: +0.086
  Min: -0.029, Max: +0.204
  Models with corr > 0: 5/6

--- Alpha sensitivity (mean SCTS across all 18 models) ---
       mean_threshold_across_classes  empirical_coverage_mondrian  mean_scts
alpha                                                                       
0.05                           0.797                        0.972     78.692
0.10                           0.733                        0.959     77.736
0.20  

## 7. Save summary JSON

In [8]:
def to_json_safe(obj):
    """Recursive numpy/tuple-key sanitizer (proven across 06v3 and 05c)."""
    if isinstance(obj, dict):
        new_dict = {}
        for k, v in obj.items():
            if isinstance(k, tuple):
                k = '|'.join(str(x) for x in k)
            elif not isinstance(k, (str, int, float, bool)) and k is not None:
                k = str(k)
            new_dict[k] = to_json_safe(v)
        return new_dict
    elif isinstance(obj, list):
        return [to_json_safe(x) for x in obj]
    elif isinstance(obj, (np.bool_, np.generic)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

summary = {
    'timestamp': datetime.now().isoformat(),
    'method': 'canonical_shared_samples_per_dataset',
    'n_datasets': len(DATASETS),
    'n_models': len(MODELS_PER_DATASET) * len(DATASETS),
    'n_samples_per_model': 1000,
    'primary_alpha': ALPHA_PRIMARY,
    'sensitivity_alphas': ALPHAS,
    'overall_stats': {
        'mean_scts': float(df_scts['scts'].mean()),
        'median_scts': float(df_scts['scts'].median()),
        'min_scts': float(df_scts['scts'].min()),
        'max_scts': float(df_scts['scts'].max()),
        'std_scts': float(df_scts['scts'].std()),
    },
    'mean_components': {
        'c1_calibration': float(df_scts['c1'].mean()),
        'c2_stability': float(df_scts['c2'].mean()),
        'c3_conformal': float(df_scts['c3'].mean()),
    },
    'mean_scts_by_dataset_architecture': df_scts.groupby(['dataset', 'architecture'])['scts'].mean().to_dict(),
    'mean_pearson_corr_scts_correctness': float(
        df_validation.groupby(['dataset', 'model'])['pearson_corr_scts_correct'].first().mean()
    ),
    'conformal_thresholds': conformal_meta,
    'alpha_sensitivity_mean': df_alpha.groupby('alpha')[['mean_threshold_across_classes', 'empirical_coverage_mondrian', 'mean_scts']].mean().to_dict(),
}

summary_safe = to_json_safe(summary)
out_path = Path(REPO) / 'results' / 'tables' / 'scts_v2_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary_safe, f, indent=2)
print(f'✓ Saved: scts_v2_summary.json')

print('\n=== Summary contents ===')
print(json.dumps(summary_safe, indent=2)[:3000])
print('... (truncated)')

✓ Saved: scts_v2_summary.json

=== Summary contents ===
{
  "timestamp": "2026-06-09T13:31:43.271058",
  "method": "canonical_shared_samples_per_dataset",
  "n_datasets": 1,
  "n_models": 6,
  "n_samples_per_model": 1000,
  "primary_alpha": 0.05,
  "sensitivity_alphas": [
    0.05,
    0.1,
    0.2
  ],
  "overall_stats": {
    "mean_scts": 78.69240813844402,
    "median_scts": 81.35485076904297,
    "min_scts": 0.5086222290992737,
    "max_scts": 99.99993133544922,
    "std_scts": 12.047176570362593
  },
  "mean_components": {
    "c1_calibration": 0.9579820374349753,
    "c2_stability": 0.5615074927372237,
    "c3_conformal": 0.9449201997698496
  },
  "mean_scts_by_dataset_architecture": {
    "nsl_kdd_v2|dnn": 76.50929688036442,
    "nsl_kdd_v2|rf": 79.19377402123808,
    "nsl_kdd_v2|xgb": 80.37415351372957
  },
  "mean_pearson_corr_scts_correctness": 0.08608197519558691,
  "conformal_thresholds": {
    "nsl_kdd_v2/rf_5class_cw": {
      "marginal_threshold_alpha_0.05": 1.0,
      "

In [9]:
import pandas as pd, glob, os
SNAP=sorted(glob.glob('/content/drive/MyDrive/XIDS_Research/_baseline_results_*'))[-1]
print('baseline snapshot:', SNAP)
for f in ['scts_v2_validation.csv','scts_v2_per_class_summary.csv']:
    p=f'{SNAP}/tables/{f}'
    if os.path.exists(p):
        print(f'\n===== BASELINE {f} (NSL rows) =====')
        df=pd.read_csv(p)
        nsl=df[df.apply(lambda r: r.astype(str).str.contains('nsl').any(), axis=1)] if 'dataset' in df.columns or True else df
        print(nsl.to_string())
    else:
        print(f'\n{f}: not in snapshot at {p}')

baseline snapshot: /content/drive/MyDrive/XIDS_Research/_baseline_results_20260609_0937

===== BASELINE scts_v2_validation.csv (NSL rows) =====
       dataset             model  scts_bin_low  scts_bin_high    n  accuracy  overall_accuracy  pearson_corr_scts_correct
0   nsl_kdd_v2      rf_5class_cw             0             25    8  0.750000             0.617                  -0.032092
1   nsl_kdd_v2      rf_5class_cw            25             50   23  0.826087             0.617                  -0.032092
2   nsl_kdd_v2      rf_5class_cw            50             75   84  0.500000             0.617                  -0.032092
3   nsl_kdd_v2      rf_5class_cw            75            101  885  0.621469             0.617                  -0.032092
4   nsl_kdd_v2     xgb_5class_cw             0             25   13  0.692308             0.638                   0.120172
5   nsl_kdd_v2     xgb_5class_cw            25             50    9  0.666667             0.638                   0.120172
6 

## 8. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/07c_scts_canonical.ipynb
!git add results/tables/scts_v2_canonical.csv
!git add results/tables/scts_v2_per_class_summary.csv
!git add results/tables/scts_v2_alpha_sensitivity.csv
!git add results/tables/scts_v2_validation.csv
!git add results/tables/scts_v2_summary.json
!git status --short
!git commit -m 'Notebook 07c: SCTS-v2 trust score with Mondrian per-class conformal (18 models, geometric mean of calibration+stability+conformal, per-class thresholds with n<30 fallback, alpha sensitivity, per-class summary)'
!git push origin main

Refresh index: 100% (371/371), done.
 M notebooks/03_nsl_calibration_v2.ipynb
 M notebooks/03d_calibration_bootstrap_cis.ipynb
 M notebooks/03e_refit_hybrid_calibrators.ipynb
 M notebooks/04_shap_analysis.ipynb
 M notebooks/04b_calibration_shap_validation.ipynb
 M notebooks/04c_shap_canonical.ipynb
 M notebooks/05c_stability_canonical.ipynb
 M notebooks/06_krishna_agreement_v3.ipynb
M  notebooks/07c_scts_canonical.ipynb
 M results/tables/krishna_agreement_canonical_summary.json
M  results/tables/scts_v2_summary.json
?? models/
?? notebooks/07c_scts_canonical_mondrian_v2.ipynb
[main bc85188] Notebook 07c: SCTS-v2 trust score with Mondrian per-class conformal (18 models, geometric mean of calibration+stability+conformal, per-class thresholds with n<30 fallback, alpha sensitivity, per-class summary)
 2 files changed, 2 insertions(+), 579 deletions(-)
 rewrite notebooks/07c_scts_canonical.ipynb (99%)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression usi

In [ ]:
import os, json
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# Check both notebooks: which has Mondrian code?
for nb_name in ['07c_scts_canonical.ipynb', '07c_scts_canonical_mondrian_v2.ipynb']:
    p = Path(REPO) / 'notebooks' / nb_name
    print(f'\n=== {nb_name} ===')
    print(f'  exists: {p.exists()}')
    if not p.exists():
        continue
    print(f'  size: {p.stat().st_size} bytes')

    with open(p) as f:
        nb = json.load(f)

    all_src = ''
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            src = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
            all_src += src + '\n'

    print(f'  total cells: {len(nb["cells"])}')
    print(f'  has mondrian_conformal_thresholds: {"mondrian_conformal_thresholds" in all_src}')
    print(f'  has component_3_mondrian: {"component_3_mondrian" in all_src}')
    print(f'  has thresh_primary (old marginal): {"thresh_primary" in all_src}')

# Also check the actual git log for last commit
import subprocess
result = subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-5'],
                       capture_output=True, text=True)
print(f'\n=== Last 5 git commits ===')
print(result.stdout)

# What did bc85188 actually change?
result = subprocess.run(['git', '-C', REPO, 'show', '--stat', 'bc85188'],
                       capture_output=True, text=True)
print(f'\n=== bc85188 contents ===')
print(result.stdout[:2000])

# Look at scts_v2_summary.json — does it have mondrian fields?
sj = Path(REPO) / 'results' / 'tables' / 'scts_v2_summary.json'
if sj.exists():
    with open(sj) as f:
        data = json.load(f)
    has_mondrian_fields = any('mondrian' in k.lower() for k in
                              (list(data.keys()) + list(data.get('conformal_thresholds', {}).get(list(data.get('conformal_thresholds', {}).keys())[0] if data.get('conformal_thresholds') else '', {}).keys())))
    print(f'\n=== Current scts_v2_summary.json ===')
    print(f'  Top-level keys: {list(data.keys())}')
    if 'conformal_thresholds' in data:
        first_model_key = list(data['conformal_thresholds'].keys())[0]
        print(f'  Sample conformal_thresholds entry for {first_model_key}:')
        print(f'    {list(data["conformal_thresholds"][first_model_key].keys())}')


=== 07c_scts_canonical.ipynb ===
  exists: True
  size: 41208 bytes
  total cells: 19
  has mondrian_conformal_thresholds: True
  has component_3_mondrian: True
  has thresh_primary (old marginal): False

=== 07c_scts_canonical_mondrian_v2.ipynb ===
  exists: True
  size: 39532 bytes
  total cells: 18
  has mondrian_conformal_thresholds: True
  has component_3_mondrian: True
  has thresh_primary (old marginal): False

=== Last 5 git commits ===
bc85188 Notebook 07c: SCTS-v2 trust score with Mondrian per-class conformal (18 models, geometric mean of calibration+stability+conformal, per-class thresholds with n<30 fallback, alpha sensitivity, per-class summary)
6258826 Notebook 07c: SCTS-v2 trust score with Mondrian per-class conformal (18 models, geometric mean of calibration+stability+conformal, per-class thresholds with n<30 fallback, alpha sensitivity, per-class summary)
4e80987 Notebook 07c: SCTS-v2 trust score on canonical samples (18 models, geometric mean of calibration+stability

In [ ]:
import pandas as pd
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
df = pd.read_csv(Path(REPO) / 'results' / 'tables' / 'scts_v2_per_class_summary.csv')

print("=== Per-dataset, per-class SCTS (Mondrian) ===\n")

for ds in ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']:
    sub = df[df['dataset'] == ds]
    print(f'\n{ds}:')
    # Aggregate across models (6 per dataset)
    pivot = sub.groupby('true_class').agg({
        'mean_scts': 'mean',
        'mean_c1': 'mean',
        'mean_c2': 'mean',
        'mean_c3': 'mean',
        'accuracy': 'mean',
        'n': 'sum',
    }).round(3)
    # Order by class index
    pivot = pivot.reindex(['Normal', 'DoS', 'Probe', 'R2L', 'U2R'])
    print(pivot)

# Specifically: the NSL rf_5class_smote R2L row (was 77.4 in marginal, accuracy=0.075)
print("\n\n=== NSL rf_5class_smote R2L: marginal vs Mondrian ===")
nsl_rf_smote = df[(df['dataset'] == 'nsl_kdd_v2') & (df['model'] == 'rf_5class_smote')]
print(nsl_rf_smote[['true_class', 'n', 'mean_scts', 'mean_c1', 'mean_c2', 'mean_c3', 'accuracy']].to_string(index=False))

=== Per-dataset, per-class SCTS (Mondrian) ===


nsl_kdd_v2:
            mean_scts  mean_c1  mean_c2  mean_c3  accuracy     n
true_class                                                      
Normal         80.912    0.995    0.544    0.994     0.960  1566
DoS            77.527    0.960    0.560    0.927     0.839  1488
Probe          78.808    0.925    0.596    0.919     0.677  1266
R2L            77.170    0.954    0.546    0.935     0.067  1278
U2R            74.612    0.914    0.534    0.896     0.169   402

unsw_nb15_v2:
            mean_scts  mean_c1  mean_c2  mean_c3  accuracy     n
true_class                                                      
Normal         68.680    0.864    0.512    0.809     0.716  1200
DoS            53.363    0.594    0.490    0.546     0.043  1200
Probe          63.279    0.837    0.400    0.820     0.713  1200
R2L            59.746    0.716    0.493    0.660     0.882  1200
U2R            58.140    0.740    0.384    0.724     0.650  1200

cic_ids2017_v

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# Load cross-architecture per-sample disagreement scores
df_disagree = pd.read_csv(Path(REPO) / 'results' / 'tables' / 'sample_disagreement_canonical.csv')
print(f"Loaded: {len(df_disagree)} rows")
print(f"Columns: {df_disagree.columns.tolist()}")
print(f"Per dataset: {df_disagree['dataset'].value_counts().to_dict()}")

# For NSL: average disagreement across the 3 architecture pairs per sample
# (rf-xgb, rf-dnn, xgb-dnn)
nsl = df_disagree[df_disagree['dataset'] == 'nsl_kdd_v2']
print(f"\nNSL: {len(nsl)} rows, pairs: {nsl['pair'].unique().tolist()}, variants: {nsl['variant'].unique().tolist()}")

# Aggregate per (variant, sample_position) across pairs
nsl_per_sample = nsl.groupby(['variant', 'sample_position', 'true_class'])['disagreement_score'].mean().reset_index()
print(f"\nPer-sample after aggregating across pairs: {len(nsl_per_sample)} rows")

# Now compare disagreement scores by class
print("\n=== NSL: Mean cross-architecture disagreement by TRUE class ===")
print("(higher disagreement = architectures disagree more about feature attribution)")
for variant in ['5class_cw', '5class_smote']:
    sub = nsl_per_sample[nsl_per_sample['variant'] == variant]
    print(f'\n{variant}:')
    class_names = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
    for c in range(5):
        mask = sub['true_class'] == c
        if mask.sum() > 0:
            print(f'  {class_names[c]:<7}: n={mask.sum():>3}, mean disagreement={sub[mask]["disagreement_score"].mean():.3f}, median={sub[mask]["disagreement_score"].median():.3f}')

# Now the KEY test: load SCTS data and compare disagreement vs accuracy
df_scts = pd.read_csv(Path(REPO) / 'results' / 'tables' / 'scts_v2_canonical.csv')

# Get NSL per-sample data with both SCTS and disagreement
# Merge on (dataset, variant, sample_position) — need to extract variant from model
df_scts['variant'] = df_scts['model'].apply(lambda m: '5class_cw' if 'cw' in m else '5class_smote')

# Just look at NSL R2L samples
print("\n\n=== NSL R2L samples: SCTS vs disagreement vs correctness ===")
for variant in ['5class_cw', '5class_smote']:
    # Get disagreement for R2L samples in this variant
    nsl_r2l_dis = nsl_per_sample[(nsl_per_sample['variant'] == variant) & (nsl_per_sample['true_class'] == 3)]

    # For each model in this variant, check correctness on these samples
    for arch in ['rf', 'xgb', 'dnn']:
        model = f'{arch}_{variant}'
        scts_r2l = df_scts[(df_scts['dataset'] == 'nsl_kdd_v2') &
                            (df_scts['model'] == model) &
                            (df_scts['true_class'] == 3)]

        if len(scts_r2l) == 0 or len(nsl_r2l_dis) == 0:
            continue

        # Merge by sample_position
        merged = scts_r2l[['sample_position', 'correct', 'scts']].merge(
            nsl_r2l_dis[['sample_position', 'disagreement_score']],
            on='sample_position'
        )

        if len(merged) > 5:
            # Does disagreement predict correctness?
            corr_dis_correct = np.corrcoef(merged['disagreement_score'], merged['correct'])[0,1]
            mean_dis_correct = merged[merged['correct']==1]['disagreement_score'].mean()
            mean_dis_wrong = merged[merged['correct']==0]['disagreement_score'].mean()

            print(f'\n  {model} (R2L samples, n={len(merged)}):')
            print(f'    Pearson(disagreement, correct) = {corr_dis_correct:+.3f}')
            print(f'    Mean disagreement when CORRECT (n={(merged["correct"]==1).sum()}):   {mean_dis_correct:.3f}')
            print(f'    Mean disagreement when WRONG   (n={(merged["correct"]==0).sum()}): {mean_dis_wrong:.3f}')
            print(f'    Interpretation: {"WRONG samples have higher disagreement (c4 could help)" if mean_dis_wrong > mean_dis_correct else "CORRECT samples have higher disagreement (c4 would HURT)"}')

Loaded: 18000 rows
Columns: ['dataset', 'variant', 'pair', 'sample_position', 'true_class', 'mean_agreement', 'disagreement_score', 'sample_FA', 'sample_RA', 'sample_SA', 'sample_SRA', 'sample_RC', 'sample_PRA']
Per dataset: {'nsl_kdd_v2': 6000, 'unsw_nb15_v2': 6000, 'cic_ids2017_v2': 6000}

NSL: 6000 rows, pairs: ['rf-xgb', 'rf-dnn', 'xgb-dnn'], variants: ['5class_cw', '5class_smote']

Per-sample after aggregating across pairs: 2000 rows

=== NSL: Mean cross-architecture disagreement by TRUE class ===
(higher disagreement = architectures disagree more about feature attribution)

5class_cw:
  Normal : n=261, mean disagreement=0.573, median=0.569
  DoS    : n=248, mean disagreement=0.607, median=0.614
  Probe  : n=211, mean disagreement=0.613, median=0.618
  R2L    : n=213, mean disagreement=0.608, median=0.610
  U2R    : n= 67, mean disagreement=0.645, median=0.648

5class_smote:
  Normal : n=261, mean disagreement=0.575, median=0.574
  DoS    : n=248, mean disagreement=0.589, median=0

In [ ]:
import subprocess
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# Show what's in last 3 commits
result = subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-5'], capture_output=True, text=True)
print("=== Last 5 commits ===")
print(result.stdout)

# Show what bc85188 actually changed
result = subprocess.run(['git', '-C', REPO, 'show', '--stat', 'bc85188'], capture_output=True, text=True)
print("\n=== bc85188 contents ===")
print(result.stdout[:2500])

# Status now (should show the Mondrian v2 file and the new CSVs)
result = subprocess.run(['git', '-C', REPO, 'status', '--short'], capture_output=True, text=True)
print("\n=== Current git status ===")
print(result.stdout)

# Verify scts_v2_summary.json has Mondrian fields
import json
sj = Path(REPO) / 'results' / 'tables' / 'scts_v2_summary.json'
if sj.exists():
    with open(sj) as f:
        data = json.load(f)
    if 'conformal_thresholds' in data:
        first_key = list(data['conformal_thresholds'].keys())[0]
        keys = list(data['conformal_thresholds'][first_key].keys())
        print(f"\n=== scts_v2_summary.json ===")
        print(f"Conformal thresholds entry keys for {first_key}:")
        print(f"  {keys}")
        if 'mondrian_thresholds_alpha_0.05' in keys:
            print("  ✓ Contains Mondrian fields (this is Mondrian-version output)")
        elif 'threshold_alpha_0.05' in keys:
            print("  ⚠ Contains old marginal field (this is OLD marginal output - needs re-commit)")

=== Last 5 commits ===
bc85188 Notebook 07c: SCTS-v2 trust score with Mondrian per-class conformal (18 models, geometric mean of calibration+stability+conformal, per-class thresholds with n<30 fallback, alpha sensitivity, per-class summary)
6258826 Notebook 07c: SCTS-v2 trust score with Mondrian per-class conformal (18 models, geometric mean of calibration+stability+conformal, per-class thresholds with n<30 fallback, alpha sensitivity, per-class summary)
4e80987 Notebook 07c: SCTS-v2 trust score on canonical samples (18 models, geometric mean of calibration+stability+conformal, alpha sensitivity, per-class summary)
8c8cfbb Session memory v10: adversarial stability sealed (05c at d7df8e7, mean Jaccard 0.552, DNN advantage 11/18 dataset-stratified, perturbation magnitudes verified, Lipschitz patterns including XGB log-odds artifact, 11 audit catches)
4a2012c Session memory v9: canonical samples + Krishna agreement sealed (04c + 06v3, 10 audit catches)


=== bc85188 contents ===
commit bc

In [ ]:
import subprocess
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# What files exist in results/tables/?
result = subprocess.run(['ls', '-la', f'{REPO}/results/tables/'], capture_output=True, text=True)
print("=== Files in results/tables/ ===")
print(result.stdout)

# Which are tracked vs untracked?
result = subprocess.run(['git', '-C', REPO, 'ls-files', 'results/tables/'], capture_output=True, text=True)
print("\n=== Tracked files in results/tables/ ===")
print(result.stdout)

# What's the SCTS file family status?
import os
from pathlib import Path
for f in sorted(Path(f'{REPO}/results/tables').glob('scts_v2*')):
    size = os.path.getsize(f)
    print(f'  {f.name}: {size} bytes')

# Run git status more verbosely on those specific files
result = subprocess.run(['git', '-C', REPO, 'status', '--porcelain', 'results/tables/'], capture_output=True, text=True)
print("\n=== Status of results/tables/ files ===")
print(result.stdout if result.stdout else "(no changes)")

=== Files in results/tables/ ===
total 7500
-rw------- 1 root root    2271 May 26 00:38 baseline_comparison.csv
-rw------- 1 root root    7478 May 29 04:18 calibration_bootstrap_cis.csv
-rw------- 1 root root    2365 May 29 04:20 calibration_brier_recomputed.csv
-rw------- 1 root root     839 May 25 23:51 cic_calibration.csv
-rw------- 1 root root    2601 May 26 01:24 cic_krishna_aggregate.csv
-rw------- 1 root root    7402 May 26 01:24 cic_krishna_full.csv
-rw------- 1 root root    4932 May 26 01:24 cic_krishna_perclass.csv
-rw------- 1 root root     899 May 26 01:24 cic_krishna_summary.csv
-rw------- 1 root root     922 May 25 23:42 cic_model_comparison.csv
-rw------- 1 root root    1508 May 26 01:07 cic_stability.csv
-rw------- 1 root root    3390 May 26 00:30 cic_top10_features.csv
-rw------- 1 root root    1967 May 29 02:03 cic_v2_calibration_perclass.csv
-rw------- 1 root root     611 May 29 02:03 cic_v2_calibration_summary.csv
-rw------- 1 root root     619 May 29 02:08 cic_v2_c

In [ ]:
from google.colab import files
import os, shutil, glob

uploaded = files.upload()  # browse for calibration_shap_krishna_stability_scts_session_memory_v11.md

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

candidates = glob.glob('/content/calibration_shap_krishna_stability_scts_session_memory_v11*.md') + \
             glob.glob(f'{REPO}/calibration_shap_krishna_stability_scts_session_memory_v11*.md')

if candidates:
    src = candidates[0]
    dst = f'{REPO}/docs/calibration_shap_krishna_stability_scts_session_memory_v11.md'
    shutil.move(src, dst)
    print(f'✓ Moved: {os.path.basename(src)} → docs/')
    print(f'  Size: {os.path.getsize(dst)} bytes (expect ~31900)')
else:
    print('✗ No v11 file found')

!git add docs/calibration_shap_krishna_stability_scts_session_memory_v11.md
!git status --short
!git commit -m 'Session memory v11: SCTS-v2 Mondrian sealed (07c at bc85188, Framing B locked, 15 audit catches, all numbers verified, APS/RAPS rationale, n<30 fallback justified, relative-vs-absolute framing in limitations)'
!git push origin main

Saving calibration_shap_krishna_stability_scts_session_memory_v11.md to calibration_shap_krishna_stability_scts_session_memory_v11.md
✓ Moved: calibration_shap_krishna_stability_scts_session_memory_v11.md → docs/
  Size: 31909 bytes (expect ~31900)
Refresh index: 100% (372/372), done.
A  docs/calibration_shap_krishna_stability_scts_session_memory_v11.md
 M notebooks/03_nsl_calibration_v2.ipynb
 M notebooks/03d_calibration_bootstrap_cis.ipynb
 M notebooks/03e_refit_hybrid_calibrators.ipynb
 M notebooks/04_shap_analysis.ipynb
 M notebooks/04b_calibration_shap_validation.ipynb
 M notebooks/04c_shap_canonical.ipynb
 M notebooks/05c_stability_canonical.ipynb
 M notebooks/06_krishna_agreement_v3.ipynb
 M results/tables/krishna_agreement_canonical_summary.json
?? models/
?? notebooks/07c_scts_canonical_mondrian_v2.ipynb
[main dbfac27] Session memory v11: SCTS-v2 Mondrian sealed (07c at bc85188, Framing B locked, 15 audit catches, all numbers verified, APS/RAPS rationale, n<30 fallback justifi

In [ ]:
from google.colab import files
import os, shutil, glob

uploaded = files.upload()  # browse for 07d_scts_calib_health.ipynb

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

candidates = glob.glob('/content/07d_scts_calib_health*.ipynb') + \
             glob.glob(f'{REPO}/07d_scts_calib_health*.ipynb')

for src in candidates:
    dst = 'notebooks/07d_scts_calib_health.ipynb'
    if src != dst:
        shutil.move(src, dst)
        print(f'✓ Moved: {os.path.basename(src)} → notebooks/')
        break

size = os.path.getsize('notebooks/07d_scts_calib_health.ipynb')
print(f'✓ Present: {size} bytes (expect ~22100)')

Saving 07d_scts_calib_health.ipynb to 07d_scts_calib_health.ipynb
✓ Moved: 07d_scts_calib_health.ipynb → notebooks/
✓ Present: 22137 bytes (expect ~22100)
